# Stage 4 — Nemotron VLM Zero-Shot Detection (commercial-license NVIDIA candidate)

Self-contained. Benchmarks `nvidia/Llama-3.1-Nemotron-Nano-VL-8B-V1` on the exact same 20
frozen Gupta test sheets, same tiling + point-in-box scoring + stitch/dedup harness as the
Molmo2 / Claude runs — so results are directly comparable.

**Why this model:** it's the one NVIDIA VLM that is actually commercially usable (NVIDIA
Nemotron Open Model License). NVIDIA's more localization-focused VLMs — LocateAnything,
Eagle 2.5, VILA — are all CC BY-NC / non-commercial, so they're out for a
production-substitution project.

**Tempered expectation, stated honestly:** Nemotron-Nano-VL is a *document-intelligence*
model (its published benchmarks are all DocVQA / OCRBench / ChartQA — no detection or
grounding benchmark). It may not emit usable bounding boxes at all. This run tells us
whether a commercially-licensed NVIDIA VLM is even viable for this task, not whether it
wins.

Uses the model's real `model.chat()` API (from its HF card), not the standard
`apply_chat_template` pattern. Runs one model — designed to run in a separate kernel in
parallel with the Molmo/Claude notebook.

## 1. Fetch all 20 test sheets from GitHub (no Drive, no upload widget)

The frozen 20-sheet test set is committed in the repo as fixtures, so we `wget` them
straight into `/content` — no `drive.mount()`, no file-picker widget.

In [1]:
from pathlib import Path

SHEETS_TEST_DIR = Path("/content/gupta_test/sheets_test"); SHEETS_TEST_DIR.mkdir(parents=True, exist_ok=True)
LABELS_TEST_DIR = Path("/content/gupta_test/labels_test"); LABELS_TEST_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/TomGeorge45/pid-opensrc-substitution/main/notebooks/stage4/gupta_test_sheets"
TEST_IDS = ["0","103","11","124","129","136","145","148","15","151",
            "157","158","159","176","188","192","194","196","216","233"]

for sid in TEST_IDS:
    !wget -q -O {SHEETS_TEST_DIR}/{sid}.jpg {BASE}/sheets/{sid}.jpg
    !wget -q -O {LABELS_TEST_DIR}/{sid}.txt {BASE}/labels/{sid}.txt

n_sheets = len(list(SHEETS_TEST_DIR.glob('*.jpg')))
print(f"Fetched {n_sheets} sheets + {len(list(LABELS_TEST_DIR.glob('*.txt')))} labels")
# catch silent 404s (wget -O writes an empty file on failure)
for sid in TEST_IDS:
    assert (SHEETS_TEST_DIR/f'{sid}.jpg').stat().st_size > 1000, f'{sid}.jpg missing/empty'
    assert (LABELS_TEST_DIR/f'{sid}.txt').stat().st_size > 0, f'{sid}.txt missing/empty'
print('All 20 sheets + labels fetched OK.')

Fetched 20 sheets + 20 labels
All 20 sheets + labels fetched OK.


## 2. GPU check + install deps (Nemotron needs timm/einops/open-clip-torch)

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers accelerate timm einops open-clip-torch

NVIDIA A100-SXM4-80GB, 81920 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 43.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00


## 3. Config + tiling / scoring / stitch harness (identical to the Molmo/Claude notebook)

In [3]:
import json, re, time
from pathlib import Path
import torch
from PIL import Image

TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1})
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def load_gt_boxes(label_path, img_w, img_h):
    if not label_path.exists():
        return []
    return [yolo_line_to_xyxy(l, img_w, img_h) for l in label_path.read_text().splitlines() if l.strip()]

def _bbox_center(bbox):
    x0, y0, x1, y1 = bbox
    return (x0 + x1) / 2, (y0 + y1) / 2

def point_in_box(point, box):
    x, y = point
    x0, y0, x1, y1 = box
    return x0 <= x <= x1 and y0 <= y <= y1

def prediction_point(pred):
    if pred.get("point") is not None:
        return tuple(pred["point"])
    return _bbox_center(pred["bbox"])

def match_predictions_to_gt(predictions, gt_boxes):
    order = sorted(range(len(predictions)),
                   key=lambda i: (predictions[i].get("confidence") is None, -(predictions[i].get("confidence") or 0)))
    gt_taken = [False] * len(gt_boxes)
    n_matched = 0
    for i in order:
        pt = prediction_point(predictions[i])
        for j, box in enumerate(gt_boxes):
            if not gt_taken[j] and point_in_box(pt, box):
                gt_taken[j] = True
                n_matched += 1
                break
    return n_matched, len(predictions), len(gt_boxes)

def precision_recall_f1(predictions, gt_boxes):
    n_matched, n_pred, n_gt = match_predictions_to_gt(predictions, gt_boxes)
    precision = n_matched / n_pred if n_pred else (1.0 if n_gt == 0 else 0.0)
    recall = n_matched / n_gt if n_gt else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "n_matched": n_matched, "n_pred": n_pred, "n_gt": n_gt}

DEDUP_DISTANCE_PX = 30

def remap_tile_to_sheet(pred, tile_origin):
    ox, oy = tile_origin
    out = dict(pred)
    x0, y0, x1, y1 = pred["bbox"]
    out["bbox"] = [x0 + ox, y0 + oy, x1 + ox, y1 + oy]
    return out

def stitch_and_dedup(tile_predictions, tile_origins, dedup_distance_px=DEDUP_DISTANCE_PX):
    sheet_preds = []
    for preds, origin in zip(tile_predictions, tile_origins):
        for p in preds:
            sheet_preds.append(remap_tile_to_sheet(p, origin))
    clusters = []
    for pred in sheet_preds:
        pt = prediction_point(pred)
        etype = pred.get("entity_type")
        placed = False
        for cluster in clusters:
            if cluster["type"] != etype:
                continue
            cx, cy = prediction_point(cluster["members"][0])
            if ((pt[0]-cx)**2 + (pt[1]-cy)**2)**0.5 <= dedup_distance_px:
                cluster["members"].append(pred)
                placed = True
                break
        if not placed:
            clusters.append({"type": etype, "members": [pred]})
    deduped = []
    for cluster in clusters:
        members = cluster["members"]
        with_conf = [m for m in members if m.get("confidence") is not None]
        best = max(with_conf, key=lambda m: m["confidence"]) if with_conf else members[0]
        deduped.append(best)
    return deduped, len(sheet_preds)

## 4. Load Nemotron + prompt + parse (real `model.chat()` API from the HF model card)

In [ ]:
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

# Same upstream transformers-v5 bug we hit with Molmo (huggingface/transformers#43883):
# Nemotron's custom trust_remote_code class predates the `all_tied_weights_keys`
# convention, and v5's loader does `getattr(self, "all_tied_weights_keys", {}).keys()`
# -> AttributeError. Class-level default (empty dict, since .keys() is called on it)
# fixes it; harmless no-op for models that set their own.
import transformers.modeling_utils as _mu
_mu.PreTrainedModel.all_tied_weights_keys = {}

NEMOTRON_MODEL_ID = "nvidia/Llama-3.1-Nemotron-Nano-VL-8B-V1"

t0 = time.time()
# attn_implementation="eager" is required: Nemotron's custom config does
# `"flash_attention" in self._attn_implementation`, which crashes when that attr is
# None (its default in current transformers). Passing a string avoids the TypeError.
nemo_model = AutoModel.from_pretrained(
    NEMOTRON_MODEL_ID, trust_remote_code=True, device_map="cuda", attn_implementation="eager"
).eval()
nemo_tokenizer = AutoTokenizer.from_pretrained(NEMOTRON_MODEL_ID)
nemo_image_processor = AutoImageProcessor.from_pretrained(NEMOTRON_MODEL_ID, trust_remote_code=True, device="cuda")
print(f"Loaded {NEMOTRON_MODEL_ID} in {time.time()-t0:.1f}s")
print("VRAM used:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

# same v3-style box-JSON prompt as the Claude run, with {W}x{H} injected
NEMOTRON_PROMPT_TEMPLATE = """You are an expert P&ID reviewer analyzing one tile cropped from a larger Piping & Instrumentation Diagram. The image is {W}x{H} pixels.

Find every discrete P&ID symbol: valves, instrument circles/bubbles, flanges, nozzles, safety devices, reducers, and other equipment icons. Do NOT count plain pipe/line segments, text labels alone, dimension arrows, or borders.

Respond with ONLY a JSON array of arrays, no other text. Each inner array is exactly: [x0, y0, x1, y1, confidence, "entity_type"] - tight boxes around just the icon, absolute pixel coordinates (top-left origin), confidence 0.0-1.0. If no symbols: []"""

def run_nemotron(image, max_new_tokens=4096):
    question = NEMOTRON_PROMPT_TEMPLATE.format(W=image.width, H=image.height)
    image_features = nemo_image_processor([image])
    gen_config = dict(max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=nemo_tokenizer.eos_token_id)
    t0 = time.time()
    response = nemo_model.chat(
        tokenizer=nemo_tokenizer, question=question, generation_config=gen_config, **image_features
    )
    return response, time.time() - t0

def parse_nemotron(text):
    # tolerant parse: prefer a fenced JSON array, else treat whole response as JSON
    fenced = re.findall(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    if fenced:
        cleaned = fenced[-1].strip()
    else:
        cleaned = text.strip()
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(data, list):
        return None, f"expected a JSON array, got {type(data).__name__}"
    detections = []
    for i, item in enumerate(data):
        if not (isinstance(item, list) and len(item) == 6):
            return None, f"item {i} malformed: {item}"
        x0, y0, x1, y1, confidence, entity_type = item
        detections.append({"bbox": [float(x0), float(y0), float(x1), float(y1)],
                           "confidence": float(confidence), "entity_type": str(entity_type)})
    return detections, None

## 5. Sanity-check on ONE tile first

Nemotron is a doc-intelligence model, not a known detector — check a single tile parses
before committing to the full 127-tile run (don't burn the whole run if it can't produce
the format at all).

In [ ]:
sample_sheet = next(SHEETS_TEST_DIR.glob("*"))
_img = Image.open(sample_sheet).convert("RGB")
_tiles = compute_tile_grid(_img.width, _img.height)
_crop = _img.crop((_tiles[0]["x0"], _tiles[0]["y0"], _tiles[0]["x1"], _tiles[0]["y1"]))

raw, latency = run_nemotron(_crop)
print(f"Latency: {latency:.1f}s")
print("Raw output (first 1200 chars):")
print(raw[:1200])

dets, err = parse_nemotron(raw)
print("\nParse:", "OK" if err is None else f"FAILED: {err}")
if dets is not None:
    print(f"{len(dets)} detections parsed")

## 6. Full run over all 20 test sheets, score — same harness/data as Molmo2 & Claude

In [ ]:
test_ids = sorted(p.stem for p in SHEETS_TEST_DIR.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"Found {len(test_ids)} test sheets (expected 20)")

nemo_totals = {"n_matched": 0, "n_pred": 0, "n_gt": 0}
nemo_parse_failures = 0
nemo_total_tiles = 0

for sheet_id in test_ids:
    img_path = next(SHEETS_TEST_DIR.glob(f"{sheet_id}.*"))
    label_path = LABELS_TEST_DIR / f"{sheet_id}.txt"
    sheet_img = Image.open(img_path).convert("RGB")
    W, H = sheet_img.size
    gt_boxes = load_gt_boxes(label_path, W, H)

    tiles = compute_tile_grid(W, H)
    tile_preds, tile_origins = [], []
    for t in tiles:
        nemo_total_tiles += 1
        crop = sheet_img.crop((t["x0"], t["y0"], t["x1"], t["y1"]))
        try:
            raw_text, _lat = run_nemotron(crop)
            dets, err = parse_nemotron(raw_text)
        except Exception as e:
            dets, err = None, str(e)
        if err:
            nemo_parse_failures += 1
            dets = []
        tile_preds.append(dets)
        tile_origins.append((t["x0"], t["y0"]))

    sheet_preds, _ = stitch_and_dedup(tile_preds, tile_origins)
    result = precision_recall_f1(sheet_preds, gt_boxes)
    for k in nemo_totals:
        nemo_totals[k] += result[k]
    print(f"{sheet_id:10s} tiles={len(tiles):2d} gt={result['n_gt']:3d} pred={result['n_pred']:3d} "
          f"matched={result['n_matched']:3d} precision={result['precision']:.2f} recall={result['recall']:.2f} f1={result['f1']:.2f}")

print(f"\n--- Overall Nemotron ({NEMOTRON_MODEL_ID}) — {len(test_ids)} sheets, {nemo_total_tiles} tiles ---")
np_ = nemo_totals["n_matched"] / nemo_totals["n_pred"] if nemo_totals["n_pred"] else 0.0
nr = nemo_totals["n_matched"] / nemo_totals["n_gt"] if nemo_totals["n_gt"] else 0.0
nf1 = 2*np_*nr/(np_+nr) if (np_+nr) else 0.0
print(f"Precision: {np_:.3f} | Recall: {nr:.3f} | F1: {nf1:.3f}")
print(f"Parse failures: {nemo_parse_failures}/{nemo_total_tiles} tiles ({100*nemo_parse_failures/nemo_total_tiles:.1f}%)")
print(f"\nProvisional pass bar (Stage4_Checklist_Status.md 3.6): F1 >= 0.70")
print("Reference comparison — Molmo2 v1 F1=0.434, Claude v1 F1=0.380 on this same setup.")

## 7. Dense-sheet tiling test — does 512+enhanced help Nemotron too? (dense sheets only)

Mirrors the Molmo2 dense-sheet experiment: run ONLY the 3 worst dense sheets (151, 216, 233)
two ways — production 1024 tiles vs 512 tiles upscaled 2x + autocontrast — to see if the
tiling fix that ~2.3x'd Molmo2 also lifts Nemotron. Reuses the already-loaded model and the
already-fetched sheets (filters to the dense 3). Upscaled coords are scaled back before
sheet remap (same verified logic as the Molmo experiment).

In [ ]:
from PIL import ImageOps

def _center(b): x0,y0,x1,y1=b; return (x0+x1)/2,(y0+y1)/2
def _pt_in(pt,box): x,y=pt; x0,y0,x1,y1=box; return x0<=x<=x1 and y0<=y<=y1
def _ppoint(p): return tuple(p["point"]) if p.get("point") is not None else _center(p["bbox"])

def _prf1(preds, gt):
    order=sorted(range(len(preds)),key=lambda i:(preds[i].get("confidence") is None,-(preds[i].get("confidence") or 0)))
    taken=[False]*len(gt); m=0
    for i in order:
        pt=_ppoint(preds[i])
        for j,box in enumerate(gt):
            if not taken[j] and _pt_in(pt,box): taken[j]=True; m+=1; break
    np_,ng=len(preds),len(gt)
    p=m/np_ if np_ else (1.0 if ng==0 else 0.0); r=m/ng if ng else 1.0
    return {"precision":p,"recall":r,"f1":2*p*r/(p+r) if (p+r) else 0.0,"n_matched":m,"n_pred":np_,"n_gt":ng}

def _dedup(sp, d=30):
    cl=[]
    for pr in sp:
        pt=_ppoint(pr); et=pr.get("entity_type"); placed=False
        for c in cl:
            if c["t"]!=et: continue
            cx,cy=_ppoint(c["m"][0])
            if ((pt[0]-cx)**2+(pt[1]-cy)**2)**0.5<=d: c["m"].append(pr); placed=True; break
        if not placed: cl.append({"t":et,"m":[pr]})
    out=[]
    for c in cl:
        wc=[x for x in c["m"] if x.get("confidence") is not None]
        out.append(max(wc,key=lambda x:x["confidence"]) if wc else c["m"][0])
    return out

def nemotron_predict(image):
    raw,_=run_nemotron(image)
    return parse_nemotron(raw)

def run_config_dense(sheet_ids, tile_size, overlap, upscale_factor=1, enhance=False, label=""):
    tot={"n_matched":0,"n_pred":0,"n_gt":0}; pf=0; nt=0
    print(f"\n=== {label} (tile={tile_size}, overlap={overlap}, upscale={upscale_factor}x, enhance={enhance}) ===")
    for sid in sheet_ids:
        img=Image.open(SHEETS_TEST_DIR/f"{sid}.jpg").convert("RGB"); W,H=img.size
        gt=load_gt_boxes(LABELS_TEST_DIR/f"{sid}.txt",W,H)
        sp=[]
        for t in compute_tile_grid(W,H,tile_size,overlap):
            nt+=1
            crop=img.crop((t["x0"],t["y0"],t["x1"],t["y1"]))
            infer=crop; scale=1.0
            if upscale_factor and upscale_factor!=1:
                infer=crop.resize((crop.width*upscale_factor,crop.height*upscale_factor),Image.LANCZOS); scale=1.0/upscale_factor
            if enhance: infer=ImageOps.autocontrast(infer)
            dets,err=nemotron_predict(infer)
            if err: pf+=1; dets=[]
            for d in dets:
                bx=d["bbox"]; d["bbox"]=[bx[0]*scale+t["x0"],bx[1]*scale+t["y0"],bx[2]*scale+t["x0"],bx[3]*scale+t["y0"]]
                if d.get("point") is not None: d["point"]=[d["point"][0]*scale+t["x0"],d["point"][1]*scale+t["y0"]]
                sp.append(d)
        r=_prf1(_dedup(sp),gt)
        for k in tot: tot[k]+=r[k]
        print(f"  {sid:6s} gt={r['n_gt']:3d} pred={r['n_pred']:3d} matched={r['n_matched']:3d} P={r['precision']:.2f} R={r['recall']:.2f} F1={r['f1']:.2f}")
    p=tot["n_matched"]/tot["n_pred"] if tot["n_pred"] else 0.0; r=tot["n_matched"]/tot["n_gt"] if tot["n_gt"] else 0.0
    f1=2*p*r/(p+r) if (p+r) else 0.0
    print(f"  OVERALL  P={p:.3f} R={r:.3f} F1={f1:.3f}  | tiles={nt} parse_fail={pf}")
    return {"label":label,"precision":p,"recall":r,"f1":f1}

# compute_tile_grid, load_gt_boxes are defined in the harness cell (section 3)
DENSE = ["151","216","233"]
dense_results = []
dense_results.append(run_config_dense(DENSE, 1024, 205, label="Nemotron / 1024 baseline"))
dense_results.append(run_config_dense(DENSE, 512, 102, upscale_factor=2, enhance=True, label="Nemotron / 512 + 2x + autocontrast"))

print("\n--- Nemotron dense-sheet comparison ---")
for r in dense_results: print(f"  {r['label']:38s} P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f}")
print("Molmo2 on same 3 sheets: 1024 F1=0.186, 512+enh F1=0.436")